# Решения: модули и импорт (пара 6)

Только для преподавателя. Урок и ДЗ. Файлы создаются через `%%writefile`, запуск — `!python`.

In [ ]:
# Те же данные, что на паре 5
PREDICTIONS = [1, 0, 1, 1, 0, 1, 1, 0, 1, 0]
LABELS =      [1, 0, 1, 0, 0, 1, 1, 0, 1, 1]


## Урок. 1. hello.py

In [ ]:
%%writefile hello.py
print("hello from file")


In [ ]:
!python hello.py

Без повторного `%%writefile` запуск `!python hello.py` печатает **старый** текст: файл на диске не изменился.

## Урок. 2–3. metrics.py (с self-check под `__main__`)

In [ ]:
%%writefile metrics.py
"""metrics — метрики классификации на списках 0/1 (пара 5)."""


def my_accuracy(preds, labels):
    """Доля верных предсказаний или None при разной длине."""
    if len(preds) != len(labels):
        return None
    if not preds:
        return 0.0
    correct = 0
    for i in range(len(preds)):
        if preds[i] == labels[i]:
            correct += 1
    return correct / len(preds)


def confusion_counts(preds, labels):
    """(tp, fp, fn, tn) для меток 0/1."""
    tp = 0
    fp = 0
    fn = 0
    tn = 0
    for i in range(len(preds)):
        p = preds[i]
        y = labels[i]
        if p == 1 and y == 1:
            tp += 1
        elif p == 1 and y == 0:
            fp += 1
        elif p == 0 and y == 1:
            fn += 1
        else:
            tn += 1
    return (tp, fp, fn, tn)


if __name__ == "__main__":
    # выполняется только при запуске: python metrics.py
    print("self-check:", my_accuracy([1, 0], [1, 1]))


In [ ]:
import metrics

assert abs(metrics.my_accuracy(PREDICTIONS, LABELS) - 0.8) < 1e-9
assert metrics.confusion_counts(PREDICTIONS, LABELS) == (5, 1, 1, 3)
print(metrics.my_accuracy(PREDICTIONS, LABELS), metrics.confusion_counts(PREDICTIONS, LABELS))


In [ ]:
%%writefile main.py
"""Отчёт по метрикам. Запуск: python main.py"""

from metrics import my_accuracy, confusion_counts

PREDICTIONS = [1, 0, 1, 1, 0, 1, 1, 0, 1, 0]
LABELS = [1, 0, 1, 0, 0, 1, 1, 0, 1, 1]

acc = my_accuracy(PREDICTIONS, LABELS)
tp, fp, fn, tn = confusion_counts(PREDICTIONS, LABELS)
print("accuracy:", acc)
print("tp fp fn tn:", tp, fp, fn, tn)


In [ ]:
!python metrics.py
!python main.py

Ожидаемо: `python metrics.py` печатает `self-check: 0.5`; `python main.py` — только accuracy и счётчики.

## Урок. 4. predict.py (с необязательным порогом и подсказкой)

In [ ]:
%%writefile predict.py
"""Запуск: python predict.py БАЛЛ [ПОРОГ]  → печатает 1 (сдал) или 0"""

import sys


def predict_pass(score, threshold):
    if score >= threshold:
        return 1
    return 0


if len(sys.argv) < 2:
    print("использование: python predict.py БАЛЛ [ПОРОГ]")
    sys.exit(1)

score = int(sys.argv[1])
threshold = 60
if len(sys.argv) >= 3:
    threshold = int(sys.argv[2])
print(predict_pass(score, threshold))


In [ ]:
!python predict.py 72 60
!python predict.py 44 60
!python predict.py 72
!python predict.py

Без проверки `len(sys.argv)` запуск без аргументов даёт `IndexError: list index out of range` на `sys.argv[1]`. Сравнение строки с числом (`"72" >= 60`) — `TypeError`.

## Урок. 5. manual_tests.py и traceback

In [ ]:
%%writefile manual_tests.py
"""Тесты модуля metrics. Запуск: python manual_tests.py"""

from metrics import my_accuracy, confusion_counts

PREDICTIONS = [1, 0, 1, 1, 0, 1, 1, 0, 1, 0]
LABELS = [1, 0, 1, 0, 0, 1, 1, 0, 1, 1]

assert abs(my_accuracy(PREDICTIONS, LABELS) - 0.8) < 1e-9, "accuracy на 10 объектах"
assert my_accuracy([1], [1, 0]) is None, "разная длина → None"
assert confusion_counts(PREDICTIONS, LABELS) == (5, 1, 1, 3), "tp fp fn tn"
assert sum(confusion_counts(PREDICTIONS, LABELS)) == len(PREDICTIONS), "сумма = число объектов"
print("All tests passed")


In [ ]:
!python manual_tests.py

Если поменять `fp` и `fn` местами, падает третья проверка:

```text
Traceback (most recent call last):
  File "manual_tests.py", line 10, in <module>
    assert confusion_counts(PREDICTIONS, LABELS) == (5, 1, 1, 3), "tp fp fn tn"
AssertionError: tp fp fn tn
```

Здесь fp = fn = 1, поэтому подмена **не** ломает тест на этих данных — хороший повод обсудить, почему тестовые данные должны различать случаи. Сильным: поменять `LABELS` так, чтобы fp ≠ fn, или сломать `tn`.

## Урок. 6. Данные как модуль

In [ ]:
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/"
    "modules/08_01_functions_recursion/data/module_datasets.py",
    "module_datasets.py",
)

from module_datasets import PREDICTIONS as P_DATA, LABELS as L_DATA

print("объектов в датасете модуля:", len(P_DATA), len(L_DATA))


---

# Домашнее задание

## ДЗ 1–2. stats_tools.py и тесты

In [ ]:
%%writefile stats_tools.py
"""stats_tools — описание и масштабирование списка чисел (пара 3)."""


def describe_numbers(values):
    """Вернуть (mean, min, max, count)."""
    if not values:
        return (0.0, 0.0, 0.0, 0)
    return (sum(values) / len(values), min(values), max(values), len(values))


def min_max_scale(values):
    """Min-max scaling списка чисел → список от 0 до 1."""
    if not values:
        return []
    lo = min(values)
    hi = max(values)
    if lo == hi:
        return [0.0] * len(values)
    result = []
    for x in values:
        result.append((x - lo) / (hi - lo))
    return result


In [ ]:
%%writefile manual_tests.py
"""Запуск: python manual_tests.py"""

from stats_tools import describe_numbers, min_max_scale

EXAM_SCORES = [40, 55, 62, 75, 88, 91, 48, 100, 33, 67]

assert describe_numbers([10, 20, 30]) == (20, 10, 30, 3), "describe на трёх числах"
assert describe_numbers(EXAM_SCORES)[3] == 10, "count = 10"
scaled = min_max_scale(EXAM_SCORES)
assert abs(min(scaled) - 0) < 1e-9 and abs(max(scaled) - 1) < 1e-9, "границы 0 и 1"
assert min_max_scale([5, 5, 5]) == [0.0, 0.0, 0.0], "одинаковые числа → нули"
print("All tests passed")


In [ ]:
!python manual_tests.py

## ДЗ 3. describe.py

In [ ]:
%%writefile describe.py
"""Запуск: python describe.py 40 55 62 75"""

import sys

from stats_tools import describe_numbers

values = []
for text in sys.argv[1:]:
    values.append(float(text))

if not values:
    print("использование: python describe.py ЧИСЛО ЧИСЛО ...")
    sys.exit(1)

mean, lo, hi, count = describe_numbers(values)
print("count", count)
print("min", lo)
print("max", hi)
print("mean", mean)


In [ ]:
!python describe.py 40 55 62 75
!python describe.py

## ДЗ 4. noisy.py

In [ ]:
%%writefile noisy.py
"""noisy — модуль с демонстрацией только при прямом запуске."""


def clip(x, low, high):
    return max(low, min(high, x))


if __name__ == "__main__":
    print("проверка clip:", clip(150, 0, 100))
    print("проверка clip:", clip(-5, 0, 100))


In [ ]:
import importlib
import noisy

importlib.reload(noisy)
print(noisy.clip(150, 0, 100))

In [ ]:
!python noisy.py

Печатало при импорте, потому что `print(...)` стоял на верхнем уровне файла: при `import` выполняется весь файл, не только `def`.

## ДЗ 5. report.py и `__pycache__`

In [ ]:
%%writefile metrics.py
"""metrics — метрики классификации на списках 0/1 (пара 5)."""


def my_accuracy(preds, labels):
    """Доля верных предсказаний или None при разной длине."""
    if len(preds) != len(labels):
        return None
    if not preds:
        return 0.0
    correct = 0
    for i in range(len(preds)):
        if preds[i] == labels[i]:
            correct += 1
    return correct / len(preds)


def confusion_counts(preds, labels):
    """(tp, fp, fn, tn) для меток 0/1."""
    tp = 0
    fp = 0
    fn = 0
    tn = 0
    for i in range(len(preds)):
        p = preds[i]
        y = labels[i]
        if p == 1 and y == 1:
            tp += 1
        elif p == 1 and y == 0:
            fp += 1
        elif p == 0 and y == 1:
            fn += 1
        else:
            tn += 1
    return (tp, fp, fn, tn)


In [ ]:
%%writefile report.py
"""Отчёт из двух своих модулей. Запуск: python report.py"""

from stats_tools import describe_numbers
from metrics import my_accuracy

SCORES = [72, 55, 88, 44, 61, 90, 77, 48, 83, 58]
PREDICTIONS = [1, 0, 1, 1, 0, 1, 1, 0, 1, 0]
LABELS = [1, 0, 1, 0, 0, 1, 1, 0, 1, 1]

mean, lo, hi, count = describe_numbers(SCORES)
print("баллы: count", count, "min", lo, "max", hi, "mean", mean)
print("accuracy:", my_accuracy(PREDICTIONS, LABELS))


In [ ]:
!python report.py
!ls __pycache__

`__pycache__` — папка с байткодом (`*.pyc`), который Python сохраняет после импорта модуля, чтобы в следующий раз загружать быстрее. Её можно удалить: при следующем импорте она создастся заново. В git её не коммитят.